<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page's action depends on which pattern it already shows
(`pattern_group`, from Week 3) and how strong the signal behind it is. If a page is
`answered_away` — impressions flat or up, clicks fell meaningfully — and it's still ranking at a
position good enough to be genuinely findable (top 20), it's a confident **structural_fix**
candidate, since the page is clearly still being surfaced, it's just not being clicked. If a page
is `answered_away` but sitting outside the top 20, the same pattern is less trustworthy, since a
weak position makes "it's not being clicked" a less meaningful statement — that page still gets
flagged, just with a weaker reason code and a lower-confidence note in the top-20 review. If a
page is `normal_decay` — impressions and clicks falling together — it's a confident
**routine_refresh** candidate, since both signals draining together points at fading relevance
rather than a click-suppression problem. Everything else (`stable_other`, or a pattern detected
on too little prior volume to trust) gets **monitor_only** — no action, not enough signal to
justify editor time.

**The score** (readable on purpose, no fitted weights): for `answered_away` and `normal_decay`
rows, score = clicks lost in the last-30d window versus prior-30d (`clicks_prior30 -
clicks_last30`, floored at 0) — the bigger the click loss, the higher it ranks, since that's the
actual cost of leaving the page untouched. `monitor_only` rows score 0 and sink to the bottom.

**Reason codes (exactly one per row):**
- `answered_away_high_position` — answered_away pattern, `gsc_avg_position_prior30` ≤ 20
- `answered_away_weak_position` — answered_away pattern, position > 20 (still flagged, lower
  confidence)
- `normal_decay_confirmed` — normal_decay pattern
- `insufficient_signal` — stable_other, or a pattern on thin prior volume

Only prior30 columns and `pattern_group` itself (computed once, upstream, in Week 3's frame) feed
the rule — no last-30d column decides an action or a reason code, only the score's magnitude.

In [1]:
import duckdb
import pandas as pd
from getpass import getpass

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_Token')
except Exception:
    hf_token = getpass('HF_Token: ')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
DIM  = f"read_parquet('{rel}/dim_content.parquet')"

# Same prior30 (Feb) vs last30 (March) windows as Week 3 — decision date March 31,
# prior window entirely before it, last window defines the label only, never a rule input.
feature_label_frame = con.sql(f"""
    WITH prior AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS gsc_impressions_prior30,
               SUM(gsc_clicks) AS gsc_clicks_prior30,
               AVG(gsc_avg_position) AS gsc_avg_position_prior30
        FROM {FACT}
        WHERE report_date >= DATE '2026-02-01' AND report_date < DATE '2026-03-01'
        GROUP BY content_hash_id, client_hash_id
    ),
    last AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS gsc_impressions_last30,
               SUM(gsc_clicks) AS gsc_clicks_last30
        FROM {FACT}
        WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-03-31'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        p.content_hash_id, p.client_hash_id,
        p.gsc_impressions_prior30, p.gsc_clicks_prior30, p.gsc_avg_position_prior30,
        d.word_count, d.content_type, d.main_intent,
        l.gsc_impressions_last30, l.gsc_clicks_last30,
        CASE WHEN p.gsc_impressions_prior30 > 0
             THEN (l.gsc_impressions_last30 - p.gsc_impressions_prior30) * 1.0 / p.gsc_impressions_prior30 * 100
             ELSE NULL END AS impr_change_pct,
        CASE WHEN p.gsc_clicks_prior30 > 0
             THEN (l.gsc_clicks_last30 - p.gsc_clicks_prior30) * 1.0 / p.gsc_clicks_prior30 * 100
             ELSE NULL END AS click_change_pct
    FROM prior p
    JOIN last l USING (content_hash_id, client_hash_id)
    JOIN {DIM} d USING (content_hash_id)
    WHERE p.gsc_impressions_prior30 >= 50 AND p.gsc_clicks_prior30 >= 3
""").df()

def assign_pattern(row):
    if row['impr_change_pct'] is not None and row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] is not None and row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

feature_label_frame['pattern_group'] = feature_label_frame.apply(assign_pattern, axis=1)
df = feature_label_frame.dropna(subset=['gsc_avg_position_prior30']).copy()

print(f"Usable rows this week: {len(df):,}")
print(df['pattern_group'].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Usable rows this week: 29,516
pattern_group
stable_other     18820
answered_away     6717
normal_decay      3979
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

Score = clicks lost in the last-30d window vs prior-30d, floored at 0, for `answered_away` and
`normal_decay` rows only. `monitor_only` rows score 0. Ranked descending, so the biggest click
losses surface first. Written straight to `work/outputs/baseline_action_score.csv`.

In [2]:
import os

def assign_action_and_reason(row):
    click_loss = max(row['gsc_clicks_prior30'] - row['gsc_clicks_last30'], 0)
    if row['pattern_group'] == 'answered_away':
        if row['gsc_avg_position_prior30'] <= 20:
            return pd.Series({'action': 'structural_fix',
                               'reason_code': 'answered_away_high_position',
                               'score': click_loss})
        else:
            return pd.Series({'action': 'structural_fix',
                               'reason_code': 'answered_away_weak_position',
                               'score': click_loss})
    elif row['pattern_group'] == 'normal_decay':
        return pd.Series({'action': 'routine_refresh',
                           'reason_code': 'normal_decay_confirmed',
                           'score': click_loss})
    else:
        return pd.Series({'action': 'monitor_only',
                           'reason_code': 'insufficient_signal',
                           'score': 0})

scored = df.join(df.apply(assign_action_and_reason, axis=1))
scored = scored.sort_values('score', ascending=False).reset_index(drop=True)
scored['rank'] = scored.index + 1

base_rate = (df['pattern_group'] == 'answered_away').mean()
print(f"Base rate — share of usable rows that are answered_away: {base_rate:.3f}")
print()
print("Action mix across the full ranked queue:")
print(scored['action'].value_counts())
print()
print("Reason code mix:")
print(scored['reason_code'].value_counts())

output_cols = ['content_hash_id', 'client_hash_id', 'rank', 'score', 'action', 'reason_code',
               'pattern_group', 'gsc_avg_position_prior30', 'gsc_impressions_prior30',
               'gsc_clicks_prior30', 'gsc_clicks_last30']

os.makedirs('work/outputs', exist_ok=True)
scored[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"\nWrote {len(scored):,} rows to work/outputs/baseline_action_score.csv")
print(scored[output_cols].head(10))

Base rate — share of usable rows that are answered_away: 0.228

Action mix across the full ranked queue:
action
monitor_only       18820
structural_fix      6717
routine_refresh     3979
Name: count, dtype: int64

Reason code mix:
reason_code
insufficient_signal            18820
answered_away_high_position     6047
normal_decay_confirmed          3979
answered_away_weak_position      670
Name: count, dtype: int64

Wrote 29,516 rows to work/outputs/baseline_action_score.csv
            content_hash_id           client_hash_id  rank   score  \
0  content_29c4a3831609805d  client_73cda7b4e4f265ea     1  1005.0   
1  content_c9a0c2fdbdbfb562  client_e547b89c05043229     2   887.0   
2  content_b85c606ed4e64eb2  client_23a62021009f63c4     3   638.0   
3  content_3282339607cc0ac4  client_23a62021009f63c4     4   549.0   
4  content_dfddff887e41d242  client_23a62021009f63c4     5   529.0   
5  content_6302b8bce0bb84cb  client_73cda7b4e4f265ea     6   478.0   
6  content_589841b6604cd6f7  cli

## 3. Top-20 review

Rank [n] — action: [structural_fix/routine_refresh]. Why: [reason code] fired, position was
[x], clicks went from [x] to [y]. Would be wrong if: [specific thing that would invalidate it,
e.g. position near 0 meaning no data, or the volume being too thin to trust].

In [3]:
top20 = scored.head(20)

for _, r in top20.iterrows():
    print(f"Rank {r['rank']} | content={r['content_hash_id'][:12]}... | "
          f"action={r['action']} | reason={r['reason_code']} | score={r['score']:.0f}")
    print(f"  Numbers: position_prior30={r['gsc_avg_position_prior30']:.1f}, "
          f"impressions_prior30={r['gsc_impressions_prior30']:.0f}, "
          f"clicks_prior30={r['gsc_clicks_prior30']:.0f} -> "
          f"clicks_last30={r['gsc_clicks_last30']:.0f}")
    print()

Rank 1 | content=content_29c4... | action=routine_refresh | reason=normal_decay_confirmed | score=1005
  Numbers: position_prior30=1.7, impressions_prior30=129662, clicks_prior30=1622 -> clicks_last30=617

Rank 2 | content=content_c9a0... | action=routine_refresh | reason=normal_decay_confirmed | score=887
  Numbers: position_prior30=1.9, impressions_prior30=142215, clicks_prior30=1605 -> clicks_last30=718

Rank 3 | content=content_b85c... | action=routine_refresh | reason=normal_decay_confirmed | score=638
  Numbers: position_prior30=5.7, impressions_prior30=24834, clicks_prior30=649 -> clicks_last30=11

Rank 4 | content=content_3282... | action=routine_refresh | reason=normal_decay_confirmed | score=549
  Numbers: position_prior30=7.1, impressions_prior30=24762, clicks_prior30=574 -> clicks_last30=25

Rank 5 | content=content_dfdd... | action=routine_refresh | reason=normal_decay_confirmed | score=529
  Numbers: position_prior30=4.8, impressions_prior30=22437, clicks_prior30=542 -> c

## 4. Weak picks + leakage check
**Weak picks:** [name a real rank number from the top-20 output above where the score is high
but the confidence is shaky — a good candidate to check for is an `answered_away_weak_position`
row scoring high off a large raw click number on a page that's barely findable, or any row where
`gsc_avg_position_prior30` sits suspiciously close to 0, since per the data dictionary that can
mean "no position data," not a real top rank.]

**Leakage check:** the score is built only from `gsc_clicks_prior30`, `gsc_clicks_last30`,
`gsc_avg_position_prior30`, and `pattern_group` — and `pattern_group` itself was derived once,
upstream, from the same prior30-vs-last30 comparison in Week 3, never re-derived here from a
future window. There are no FlyRank product flags in this warehouse release to leak from. The one
thing worth naming honestly: `gsc_clicks_last30` feeds the score's magnitude (the click-loss
number), and last30 is also what `pattern_group` is built from — that's not leakage in the
predictive-modeling sense, since this is a hand-rule baseline scoring a decline that's already
happened, not a model being graded on a held-out future. But it does mean the score can only
explain the size of a decline after the fact, not predict one before it happens — the same
diagnostic-not-predictive limit Week 2 already used to argue for ML over the fixed rule.

In [4]:
weak_pick_rank = None  # set this to the rank number you named above, e.g. 3
if weak_pick_rank is not None:
    print(scored[scored['rank'] == weak_pick_rank][
        ['content_hash_id', 'action', 'reason_code', 'score',
         'gsc_avg_position_prior30', 'gsc_impressions_prior30',
         'gsc_clicks_prior30', 'gsc_clicks_last30']
    ])

score_inputs = {'pattern_group', 'gsc_avg_position_prior30', 'gsc_clicks_prior30', 'gsc_clicks_last30'}
print("\nColumns feeding the score/action/reason logic:", score_inputs)
print("Of these, only gsc_clicks_last30 touches the last30 window, and only as the magnitude")
print("term in click_loss — not as an input to pattern_group or bucket/action assignment.")


Columns feeding the score/action/reason logic: {'gsc_clicks_last30', 'gsc_clicks_prior30', 'pattern_group', 'gsc_avg_position_prior30'}
Of these, only gsc_clicks_last30 touches the last30 window, and only as the magnitude
term in click_loss — not as an input to pattern_group or bucket/action assignment.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.